# 🤖 RoboCraft - Google Colab 3D GPU Server (%100 Ücretsiz & Kotasız)

Bu notebook, **RoboCraft** uygulamanız için Google'ın ücretsiz **16GB Nvidia T4 GPU** donanımını kullanarak 3D robot modelleri üreten Stability AI motorunu çalıştırır.

### 📌 Kullanım Adımları:
1. Üst menüden **Runtime (Çalışma Zamanı) > Change runtime type** diyerek **T4 GPU** seçildiğinden emin olun.
2. Aşağıdaki 2 hücreyi sırayla çalıştırın.
3. En alttaki hücrede beliren **** bağlantısını kopyalayın.
4. RoboCraft arayüzündeki **Settings** sayfasına bu linki yapıştırın ve kotasız model üretmeye başlayın!


In [ ]:
# [1] 3D Motorunu Kur (TripoSR - Stability AI) ~ 20 saniye sürer
import os

!pip install -q --no-build-isolation git+https://github.com/tatsy/torchmcubes.git
!pip install -q einops omegaconf trimesh rembg onnxruntime-gpu gradio gradio_client

if not os.path.exists("/content/TripoSR"):
    !git clone https://github.com/VAST-AI-Research/TripoSR.git
%cd /content/TripoSR

print("
✅ TripoSR 3D Motoru ve kütüphaneler başarıyla kuruldu! (Sıfır Hata)")

In [ ]:
# [2] RoboCraft 3D GPU Sunucusunu Başlat
import gradio as gr
import torch
import tempfile
from PIL import Image
import rembg
from tsr.system import TSR
from tsr.utils import remove_background, resize_foreground, to_gradio_3d_orientation

print("🚀 3D Model GPU uzerine yukleniyor...")
device = "cuda:0" if torch.cuda.is_available() else "cpu"
model = TSR.from_pretrained(
    "stabilityai/TripoSR",
    config_name="config.yaml",
    weight_name="model.ckpt",
)
model.renderer.set_chunk_size(8192)
model.to(device)
rembg_session = rembg.new_session()

def generate_3d(input_image):
    if input_image is None:
        raise gr.Error("Resim bulunamadi!")
    processed = remove_background(input_image.convert("RGB"), rembg_session)
    processed = resize_foreground(processed, 0.85)
    with torch.no_grad():
        scene_codes = model(processed, device=device)
    mesh = model.extract_mesh(scene_codes, True, resolution=256)[0]
    mesh = to_gradio_3d_orientation(mesh)
    tmp_file = tempfile.NamedTemporaryFile(suffix=".glb", delete=False)
    mesh.export(tmp_file.name)
    return tmp_file.name

demo = gr.Interface(
    fn=generate_3d,
    inputs=gr.Image(type="pil", label="Giris Gorseli"),
    outputs=gr.File(label="model.glb"),
    title="RoboCraft AI 3D Server",
)

print("
🚀 3D Sunucu Baslatildi! Asagidaki https://xxxx.gradio.live linkini kopyalayip RoboCraft Settings sayfasina yapistirin:
")
demo.launch(share=True)